# 07 - Voir les augmentations des quatre pipelines cote a cote

Une ligne par pipeline, une colonne par tirage aleatoire, sur la meme tuile. Chaque ligne execute le **vrai** code d'augmentation du framework : `YOLODataset` d'Ultralytics (mosaique comprise), les transforms albumentations de `aug_config` pour RF-DETR, les classes `torchvision.transforms.v2` de la liste d'ops des depots pour RT-DETR/D-FINE, et `random_affine` + `augment_hsv` de YOLOX orchestres comme sa `MosaicDetection`.

Les quatre cohabitent dans ce runtime parce qu'aucun entrainement n'a lieu : YOLOX est clone **sans etre installe**, seules ses fonctions numpy/cv2 sont importees.

In [ ]:
# --- Installation (aucun entrainement ici : les 4 pipelines coexistent) ---
!pip install -q ultralytics albumentations loguru thop tabulate psutil pycocotools

In [ ]:
# --- Connexion Drive ---
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- Code du benchmark (package aphids_det) ---
# Depot public : rien a regler. S'il repassait en prive, deposer un jeton GitHub
# dans les secrets Colab (icone cle a gauche) sous le nom GITHUB_TOKEN.
import os, subprocess, sys

REPO_DIR = "/content/aphids_detection"
REPO_URL = "https://github.com/EmmaDub/aphids_detection.git"
os.environ["GIT_TERMINAL_PROMPT"] = "0"   # echouer net, plutot qu'attendre un mot de passe

url = REPO_URL
try:
    from google.colab import userdata
    jeton = userdata.get("GITHUB_TOKEN")
    if jeton:
        url = REPO_URL.replace("https://", f"https://{jeton}@")
except Exception:
    pass

if os.path.exists(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", url],
                   capture_output=True, text=True)
    r = subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], capture_output=True, text=True)
else:
    r = subprocess.run(["git", "clone", "-q", url, REPO_DIR], capture_output=True, text=True)

if r.returncode != 0:
    raise RuntimeError("Recuperation du code impossible :\n"
                       + (r.stderr or r.stdout or "").replace(url, REPO_URL).strip()
                       + "\n\nDepot prive ? Voir la section Demarrage du README.")

sys.path.insert(0, REPO_DIR)
import aphids_det
print("aphids_det", aphids_det.__version__, "|", REPO_DIR)

In [ ]:
# --- CONFIG DONNEES (source des tuiles + variante labels_cell_20) ---
# Memes chemins que comparaison_modeles_ultralytics.ipynb : c'est ici, et nulle
# part ailleurs, qu'on change de jeu de donnees.
from pathlib import Path
import aphids_det.config as cfg

cfg.BASE_DIR = Path("/content/drive/MyDrive/Emma/puceron_model_2026/"
                    "puceron_model_E2026_3/data/tuile_viz02_640_128")
cfg.SPLIT_DIR = cfg.BASE_DIR / "split"
cfg.CELL_DIR = Path("/content/drive/MyDrive/Emma/puceron_model_2026/"
                    "puceron_model_E2026_2/data/cell/tuile_viz02_640_128_cell")
cfg.MAIN_IMAGES_DIR = cfg.BASE_DIR / "images"
cfg.BG_DIR = Path("/content/drive/MyDrive/Emma/puceron_model_2026/"
                  "puceron_model_E2026_2/data/images_complete")
cfg.BG_ZIP = cfg.BG_DIR.parent / "tuile_viz02_640_128_background.zip"

cfg.EXPERIMENT_NAME = "yolo_neg1"
cfg.SEARCH_VARIANT = "labels_cell_20"
cfg.VARIANTS = {
    "labels_cell_20": cfg.V(
        cfg.SPLIT_DIR / "split_assignments_all_background.csv", "labels_visible_20",
        extra_train={"csv":        cfg.CELL_DIR / "split" / "split_assignments.csv",
                     "labels_dir": cfg.CELL_DIR / "labels_cell_20",
                     "images_dir": cfg.CELL_DIR / "images_lookmatched3"}),
}

cfg.CLASS_NAMES = {0: "Apterous_aphid", 1: "Alate_aphid"}
cfg.N_CV_FOLDS = 5          # folds 0-4 en validation croisee
cfg.TEST_FOLD = 5           # fold 5 : test, jamais utilise ici
cfg.CV_FOLDS = list(range(cfg.N_CV_FOLDS))
cfg.NEG_RATIO = 3

# --- SORTIES (Drive) et budget ---
cfg.OUT_DIR = Path("/content/drive/MyDrive/Emma/puceron_model_2026/"
                   "puceron_model_article/data")
cfg.EPOCHS = 30
cfg.IMGSZ = 640
cfg.PATIENCE = 5
cfg.SEED = 42
cfg.USE_WANDB = True
cfg.WANDB_PROJECT = "comparaison_pucerons_detection"

cfg.refresh()
cfg.summary()

In [ ]:
# --- Verification des chemins Drive avant de lancer quoi que ce soit ---
attendus = {
    "tuiles (images)": cfg.MAIN_IMAGES_DIR,
    "labels": cfg.VARIANTS[cfg.SEARCH_VARIANT]["labels_dir"],
    "CSV de split": cfg.VARIANTS[cfg.SEARCH_VARIANT]["csv"],
    "fonds (images_complete)": cfg.BG_DIR,
    "renfort cell : images": cfg.VARIANTS[cfg.SEARCH_VARIANT]["extra_train"]["images_dir"],
    "renfort cell : labels": cfg.VARIANTS[cfg.SEARCH_VARIANT]["extra_train"]["labels_dir"],
    "renfort cell : CSV": cfg.VARIANTS[cfg.SEARCH_VARIANT]["extra_train"]["csv"],
    "sorties (Drive)": cfg.OUT_DIR,
}
for nom, p in attendus.items():
    p = Path(p)
    etat = "OK     " if p.exists() else "MANQUE "
    extra = ""
    if p.is_dir():
        try:
            extra = f"  ({sum(1 for _ in p.iterdir())} entrees)"
        except OSError:
            pass
    print(f"{etat}{nom:28s} {p}{extra}")
if not Path(cfg.BG_DIR).exists():
    print(f"\n(fonds absents : l'archive {cfg.BG_ZIP} sera extraite en local)")

In [ ]:
# --- Construction des folds (symlinks locaux, a refaire a chaque session Colab) ---
# Les listes de train figees sur le Drive contiennent des chemins absolus ecrits
# par la session qui les a creees : ils sont automatiquement regreffes sur la
# racine de folds courante, la selection de tuiles reste donc identique.
from aphids_det import folds
folds.build_folds()

In [ ]:
# --- Diagnostic : a lancer si un fold parait vide ---
# Affiche images/labels par fold, et ce que donnent les listes figees du Drive
# une fois regreffees sur la racine courante.
folds.diagnose(fold=0)

## Figure comparative

A regarder en priorite : la **mosaique** (presente pour YOLO et YOLOX, absente des trois DETR), les **miroirs verticaux** (ajoutes par le benchmark a RT-DETR, D-FINE et YOLOX), l'amplitude de la variation **HSV**, et ce que `RandomZoomOut` + `RandomIoUCrop` produisent chez les DETR a la place de `translate` et `scale`.

In [ ]:
from aphids_det import visualize
fig = visualize.compare(fold=0, n_aug=4, seed=0, save=True)

## Le meme effet, pousse a sa valeur extreme

La figure precedente montre des **tirages aleatoires** : deux lignes ne sont donc jamais comparables case par case. Celle-ci est **deterministe** -- une colonne = un effet pousse a la borne de la reference (saturation x1.2, echelle x0.75 et x1.25, translation +10 %...), applique par le code de chaque framework. C'est la figure a utiliser pour verifier qu'un meme reglage produit bien le meme effet partout.

Le determinisme n'est pas obtenu en trichant sur le calcul : quand l'API le permet on passe un intervalle degenere (`saturation=(1.2, 1.2)`, `scales=(1.25, 1.25)`), sinon on force les tirages a leur borne. Quand un framework n'expose pas le reglage, la case montre ce qu'il fait **a la place** -- recadrage `RandomIoUCrop` pousse a son decalage maximal pour RT-DETR et D-FINE, branche recadrage du `OneOf` interne pour RF-DETR -- avec une legende qui le precise. Elle ne reste vide que faute d'alternative, comme la mosaique.

In [ ]:
visualize.compare_effets(fold=0, save=True)

## Plusieurs tuiles

Meme figure sur d'autres tuiles : une seule tuile ne suffit pas a juger d'une augmentation aleatoire.

In [ ]:
tuiles, vivier = visualize.sample_tiles(fold=0, n=3, seed=1, min_boxes=2)
for t in tuiles:
    visualize.compare(fold=0, tile=t, pool=vivier, n_aug=4, seed=1, save=True)

## Un seul pipeline, plus de tirages

Pour inspecter un framework en particulier.

In [ ]:
visualize.compare(fold=0, n_aug=6, seed=2,
                  pipelines={"YOLOX-Nano": visualize.PIPELINES["YOLOX-Nano"]})

## Comparer les trois geometries possibles pour RT-DETR / D-FINE

`cfg.DETR_GEOM` decide de l'amplitude d'echelle et de translation des deux DETR :

- **`reference`** (defaut) : les transforms natives des depots, bornes calees sur `scale = 0.25` -> objet dans x[0.75, 1.25], comme les YOLO ;
- **`affine`** : `RandomZoomOut` et `RandomIoUCrop` remplaces par un `RandomAffine` portant exactement `translate = 0.1` et `scale = (0.75, 1.25)` ;
- **`natif`** : les amplitudes d'origine des depots, x[0.25, 3.3].

La figure ci-dessous met les trois cote a cote sur la meme tuile. Choisir, puis fixer `cfg.DETR_GEOM` dans la cellule de configuration des notebooks d'entrainement.

In [ ]:
def geometrie(mode):
    def pipeline(tile, n_aug, pool, seed=0):
        cfg.DETR_GEOM = mode          # lu a chaque appel par patch_ops
        return visualize.aug_detr(tile, n_aug, pool, seed=seed)
    return pipeline

visualize.compare(fold=0, n_aug=4, seed=0, pipelines={
    "RT-DETR / D-FINE (reference)": geometrie("reference"),
    "RT-DETR / D-FINE (affine)":    geometrie("affine"),
    "RT-DETR / D-FINE (natif)":     geometrie("natif"),
})
cfg.DETR_GEOM = "reference"           # on remet le defaut

In [ ]:
# --- Table des correspondances, a mettre en regard de la figure ---
from aphids_det import augment
augment.table()